# Rainfed wheat
This notebook runs 2000–2009 with the user-facing `CropSimulation` API.

In [ ]:
import Pkg
Pkg.activate(normpath(joinpath(@__DIR__, "..")))

using Agrocosm
import JLD2: @load

cdev = identity  # use CuArray on an NVIDIA GPU
FT = Float32     # change to Float64 for double precision

In [ ]:
notebook_dir = @__DIR__
initial_file = joinpath(notebook_dir, "initial_wheat.jld2")
climate_files = [
    joinpath(notebook_dir, "climate_2000_2009.jld2"),
]
@load initial_file initial_data

simulation = initialize_simulation(
    cft1, initial_data;
    indices = [1],
    device = cdev,
    T = FT,
    days = 10 * 365,
    auto_fertilizer = false,
)
run_simulation!(simulation, climate_files)
simulation_summary(simulation)

In [ ]:
@assert simulation.simulated_days == 10 * 365
@assert eltype(simulation.output.crop.npp) === FT
@assert all(isfinite, Array(simulation.output.crop.npp))

diagnostic_summary = (
    water_residual = maximum(abs, Array(simulation.water_balance.residual)),
    nitrogen_residual = maximum(abs, Array(simulation.nitrogen_balance.residual)),
    carbon_residual = maximum(abs, Array(simulation.carbon_balance.residual)),
    thermal_residual = maximum(abs, Array(simulation.thermal_balance.energy_residual)),
)

In [ ]:
using Plots
gpp = vec(Array(simulation.output.crop.gpp))
npp = vec(Array(simulation.output.crop.npp))
x = 1:length(gpp)
p = plot(x, gpp;
    label = "GPP",
    color = "#2878B5",
    linewidth = 2,
    size = (2000, 500),
    title = "Crop GPP and NPP",
    xlabel = "Time step (day)",
    ylabel = "Productivity (gC/m^2/day)",

    titlefontsize = 16,
    guidefontsize = 14,   
    tickfontsize = 12,   
    legendfontsize = 11,
    
    left_margin   = (14, :mm),
    bottom_margin = (14, :mm),
    right_margin  = (6, :mm),
    top_margin    = (6, :mm),

    framestyle = :box,
    background_color = :white,
    foreground_color = :black,
    grid = true,
    gridalpha = 0.25,
)

plot!(p, x, npp;
    label = "NPP",
    color = "#E45756",
    linewidth = 2
)

display(p)
savefig(p, "crop_gpp_npp.png")